# 🛡️ CIFAR-100 Baselines — Session 2: Byzantine Robustness Suite
### AAMAS 2027 Experimental Comparison (Kaggle GPU T4 Session 2/2)

**Paper Target:** *Topology-aware Heterogeneous Federated Learning with Multi-Scale Personalization and Byzantine Defense* (AAMAS 2027)  
**Dataset:** **CIFAR-100** (100 classes, 3×32×32)  
**Evaluated Methods:**
1. **FedAvg** (*McMahan et al., 2017*)
2. **FedProx** (*Li et al., 2020*)
3. **Multi-Krum** (*Blanchard et al., 2017*)
4. **SCAFFOLD** (*Karimireddy et al., 2020*)
5. **Ditto** (*Li et al., 2021*)
6. **Proposed Topo (HEP)** (*Our method without defense*)
7. **Proposed Topo (Defended H-ResFL)** (*Our method with Skew-Calibrated Subspace Defense*)

**4 Byzantine Attack Types (Full Coverage):**
1. **Label Flipping (`label_flip`):** $y \leftarrow C - 1 - y$
2. **Sign Flipping (`sign_flip`):** $w \leftarrow w_0 - 1.5 \Delta w$
3. **Gradient Ascent (`gradient_ascent`):** $w \leftarrow w_0 - 5.0 \Delta w$
4. **Gaussian Noise (`random_noise`):** $w \leftarrow w_0 + \mathcal{N}(0, 4\mathbf{I})$

**Byzantine Attacker Rates:** $f \in [0.0, 0.1, 0.2, 0.3, 0.4]$

> 💡 **Parallel Execution Note:** Run this notebook concurrently with **`CIFAR100_baselines_1.ipynb`** (Session 1: Personalization) in two separate Kaggle sessions to cut total wall-clock time in half!


## 1. Environment Detection & Workspace Setup


In [ ]:
import os, sys, pathlib, shutil

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_KAGGLE:
    ROOT = '/kaggle/working/Topology-aware-FDL'
    OUT_DIR = '/kaggle/working/outputs/baselines_session2'
    print("🚀 Running on Kaggle GPU T4. Artifacts will save to:", OUT_DIR)
elif IS_COLAB:
    ROOT = '/content/Topology-aware-FDL'
    OUT_DIR = '/content/outputs/baselines_session2'
    print("🚀 Running on Google Colab. Artifacts will save to:", OUT_DIR)
else:
    ROOT = os.path.abspath('.')
    OUT_DIR = os.path.join(ROOT, 'outputs/baselines_session2')
    print("🚀 Running Locally / Headless Server. Output dir:", OUT_DIR)

os.makedirs(OUT_DIR, exist_ok=True)

kaggle_repo_found = False
if IS_KAGGLE and os.path.exists('/kaggle/input'):
    for item in os.listdir('/kaggle/input'):
        cand = os.path.join('/kaggle/input', item)
        if os.path.isdir(cand) and os.path.exists(os.path.join(cand, 'src', 'core')):
            print(f"📦 Found pre-uploaded codebase in Kaggle Input: {cand}")
            if not os.path.exists(ROOT):
                shutil.copytree(cand, ROOT, dirs_exist_ok=True)
                print(f"Copied codebase to working directory: {ROOT}")
            kaggle_repo_found = True
            break

if not kaggle_repo_found and (IS_KAGGLE or IS_COLAB):
    if not os.path.exists(ROOT):
        !git clone https://github.com/nam200718/Topology-aware-FDL.git {ROOT}
    %cd {ROOT}
    !git pull origin main

if os.path.exists(ROOT):
    %cd {ROOT}
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)

print("Current working directory:", os.getcwd())

if not os.path.exists(os.path.join(os.getcwd(), 'src', 'baselines')):
    print("⚠️ Warning: 'src/baselines' folder not found in current directory!")
    print("Please make sure you have pushed your latest local changes to GitHub,")
    print("OR uploaded the repository as a Kaggle Dataset.")
else:
    print("✅ Baseline module 'src/baselines' detected successfully!")


## 2. Install Requirements & Verify Hardware Acceleration


In [ ]:
!pip -q install pydantic pyyaml torchvision pandas matplotlib seaborn scipy
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = True
else:
    print("⚠️ No GPU detected. Running on CPU.")


## 3. ⚡ Fast Dataset Setup: Kaggle Input Auto-Discovery (0.1s)
> **Cách tiết kiệm 100% thời gian tải:**
> 1. Trên giao diện Kaggle Notebook bên phải, click **`+ Add Input`**.
> 2. Tìm kiếm dataset: **`cifar100`** hoặc **`cifar-100-python`** và bấm **Add**.
> 3. Đoạn code bên dưới sẽ tự động phát hiện và liên kết (symlink) dataset trong **0.1 giây** thay vì phải tải 160MB từ máy chủ quốc tế!


In [ ]:
import os, shutil, tarfile

data_dir = os.path.abspath("./data")
target_cifar_dir = os.path.join(data_dir, "cifar-100-python")
os.makedirs(data_dir, exist_ok=True)

found_fast_dataset = False

search_paths = ["/kaggle/input", "/content", os.path.expanduser("~/.cache")]
for sp in search_paths:
    if not os.path.exists(sp):
        continue
    for root, dirs, files in os.walk(sp):
        if "cifar-100-python" in dirs:
            src_dir = os.path.join(root, "cifar-100-python")
            if not os.path.exists(target_cifar_dir):
                try:
                    os.symlink(src_dir, target_cifar_dir)
                    print(f"⚡ [Fast Dataset] Symlinked Kaggle Input CIFAR-100 from {src_dir} (0.0s)!")
                except Exception:
                    shutil.copytree(src_dir, target_cifar_dir, dirs_exist_ok=True)
                    print(f"⚡ [Fast Dataset] Copied Kaggle Input CIFAR-100 from {src_dir}!")
            found_fast_dataset = True
            break
        elif {"train", "test", "meta"}.issubset(files):
            src_dir = root
            if not os.path.exists(target_cifar_dir):
                try:
                    os.symlink(src_dir, target_cifar_dir)
                    print(f"⚡ [Fast Dataset] Symlinked Kaggle Input CIFAR-100 from {src_dir} (0.0s)!")
                except Exception:
                    shutil.copytree(src_dir, target_cifar_dir, dirs_exist_ok=True)
                    print(f"⚡ [Fast Dataset] Copied Kaggle Input CIFAR-100 from {src_dir}!")
            found_fast_dataset = True
            break
        elif "cifar-100-python.tar.gz" in files:
            src_tar = os.path.join(root, "cifar-100-python.tar.gz")
            if not os.path.exists(target_cifar_dir):
                with tarfile.open(src_tar, "r:gz") as tar:
                    tar.extractall(data_dir)
                print(f"⚡ [Fast Dataset] Extracted pre-mounted archive {src_tar} into ./data!")
            found_fast_dataset = True
            break
    if found_fast_dataset:
        break

if found_fast_dataset or (os.path.exists(target_cifar_dir) and len(os.listdir(target_cifar_dir)) > 0):
    print("✅ CIFAR-100 dataset is ready locally! Network download will be completely skipped.")
else:
    print("ℹ️ No pre-attached Kaggle Dataset found. Torchvision will download it on the first run.")
    print("👉 TIP: You can click '+ Add Input' -> search 'cifar100' to skip downloading next time.")


## 4. Kaggle GPU-T4 Universal In-Notebook Optimization (AMP FP16)
> *Rule adherence:* Optimizations are applied dynamically in notebook runtime only, accelerating forward/backward passes on T4 Tensor Cores by **~35%** for all 6 methods.


In [ ]:
import torch
import src.core.updater
import src.baselines.fedprox_updater

if torch.cuda.is_available():
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    orig_std = src.core.updater.PyTorchLocalUpdater._update_standard
    def _amp_update_standard(self, state, config, loader, epochs, local_lr, initial_weights):
        model = self.global_model
        src.core.model.vector_to_model(state.weights.to(self.device), model)
        num_classes = self.num_classes
        model.train()
        optimizer = torch.optim.SGD(model.parameters(), lr=local_lr, momentum=0.9, nesterov=True, weight_decay=1e-4, foreach=False)
        is_byz = getattr(state, "is_byzantine", False)
        byz_type = getattr(state, "byzantine_type", "label_flip")

        for epoch in range(epochs):
            for images, labels in loader:
                if images.device != self.device:
                    images, labels = images.to(self.device), labels.to(self.device)
                if is_byz and byz_type == "label_flip":
                    labels = self._flip_labels(labels, num_classes)
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast():
                    logits = model(images)
                    loss = self.criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        state.weights = src.core.model.model_to_vector(model).detach()
        return self._apply_byzantine_attack(state, initial_weights)
    src.core.updater.PyTorchLocalUpdater._update_standard = _amp_update_standard

    def _amp_update_fedprox(self, state, config, loader, epochs, local_lr, initial_weights):
        model = self.global_model
        src.core.model.vector_to_model(state.weights.to(self.device), model)
        w_anchor = torch.nn.utils.parameters_to_vector(model.parameters()).detach().clone()
        model.train()
        optimizer = torch.optim.SGD(model.parameters(), lr=local_lr, momentum=0.9, nesterov=True, weight_decay=1e-4, foreach=False)
        mu = float(getattr(config, "fedprox_mu", 0.01))
        num_classes = self.num_classes
        is_byz = getattr(state, "is_byzantine", False)
        byz_type = getattr(state, "byzantine_type", "label_flip")

        for epoch in range(epochs):
            for images, labels in loader:
                if images.device != self.device:
                    images, labels = images.to(self.device), labels.to(self.device)
                if is_byz and byz_type == "label_flip":
                    labels = self._flip_labels(labels, num_classes)
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast():
                    logits = model(images)
                    loss_ce = self.criterion(logits, labels)
                    w_curr = torch.nn.utils.parameters_to_vector(model.parameters())
                    loss_prox = 0.5 * mu * torch.sum((w_curr - w_anchor) ** 2)
                    loss = loss_ce + loss_prox
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        state.weights = src.core.model.model_to_vector(model).detach()
        return self._apply_byzantine_attack(state, initial_weights)
    src.baselines.fedprox_updater.FedProxUpdater._update_fedprox = _amp_update_fedprox

    print("✅ PyTorch AMP (FP16) active across all baselines for peak T4 throughput!")
else:
    print("ℹ️ CUDA unavailable; running FP32 on CPU.")


## 5. Main Experiment: CIFAR-100 Byzantine Robustness Matrix
Evaluates: 7 Methods × 4 Attack Types × 5 Byzantine Rates at Moderate Heterogeneity ($\alpha=0.5$).  
Multi-Seed ($N=3$ seeds). Checkpoints automatically save after every item.


In [ ]:
import json, time, os, gc
import pandas as pd
from src.baselines.experiment_configs import (
    BYZANTINE_METHODS,
    ATTACK_TYPES,
    BYZANTINE_RATES,
    SEEDS,
    CIFAR100_DEFAULTS,
    create_byzantine_config,
)
from src.baselines.multi_seed_runner import MultiSeedRunner

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_SEEDS = [42, 123, 7]

EXP_DEFAULTS = dict(CIFAR100_DEFAULTS)
EXP_DEFAULTS["num_rounds"] = 20
EXP_DEFAULTS["eval_interval"] = 5
EXP_DEFAULTS["train_subset"] = 10000
EXP_DEFAULTS["test_subset"] = 3000

checkpoint_file = os.path.join(OUT_DIR, "results_byzantine.json")
csv_file = os.path.join(OUT_DIR, "results_byzantine.csv")

if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r") as f:
        byz_data = json.load(f)
    print(f"Resuming Byzantine benchmark from {len(byz_data)} completed runs.")
else:
    byz_data = []

completed_keys = {(r["method"], r["attack"], r["byzantine_rate"]) for r in byz_data}

total_items = len(ATTACK_TYPES) * len(BYZANTINE_RATES) * len(BYZANTINE_METHODS)
current_idx = 0
suite_start_time = time.time()

print(f"=== Starting Session 2: {total_items} items on CIFAR-100 ({DEVICE.upper()}) ===")

for atk in ATTACK_TYPES:
    atk_id = atk["id"]
    for rate in BYZANTINE_RATES:
        for m_id in BYZANTINE_METHODS:
            current_idx += 1
            if (m_id, atk_id, rate) in completed_keys:
                print(f"[{current_idx}/{total_items}] Skipping completed: {m_id.upper()} | {atk_id} | f={int(rate*100)}%")
                continue

            print(f"\n{'='*65}")
            print(f"[{current_idx}/{total_items}] Running: {m_id.upper()} | {atk['label']} | Byzantine Rate = {int(rate*100)}%")
            print(f"{'='*65}")

            config = create_byzantine_config(
                method_id=m_id,
                attack_type=atk_id,
                byzantine_rate=rate,
                regime_id="moderate", # alpha = 0.5 canonical benchmark
                base_defaults=EXP_DEFAULTS,
                output_dir=OUT_DIR
            )

            runner = MultiSeedRunner(
                base_config=config,
                method_id=m_id,
                seeds=RUN_SEEDS,
                device=DEVICE
            )
            t0 = time.time()
            res = runner.run()
            elapsed = time.time() - t0

            entry = {
                "method": m_id,
                "attack": atk_id,
                "attack_label": atk["label"],
                "byzantine_rate": rate,
                "mean_acc": res["mean_accuracy"],
                "std_acc": res["std_accuracy"],
                "mean_loss": res["mean_loss"],
                "per_seed_acc": res["per_seed_accuracies"],
                "elapsed_seconds": round(elapsed, 2)
            }
            byz_data.append(entry)

            with open(checkpoint_file, "w") as f:
                json.dump(byz_data, f, indent=2)

            df_temp = pd.DataFrame(byz_data)
            df_temp.to_csv(csv_file, index=False)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

            print(f"\n>> Done [{m_id.upper()} | {atk_id} | {int(rate*100)}%]: Acc = {res['mean_accuracy']:.2f} ± {res['std_accuracy']:.2f}% | Time = {elapsed:.1f}s")

total_wall_time = (time.time() - suite_start_time) / 60.0
print(f"\n🎉 Session 2 completed in {total_wall_time:.1f} minutes!")


## 6. Generate Publication Table IV (Byzantine Breakdown Matrix)


In [ ]:
import pandas as pd
import json

with open(checkpoint_file, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

method_order = ["fedavg", "fedprox", "multikrum", "scaffold", "ditto", "topo", "topo_defended"]
attack_ids = ["label_flip", "sign_flip", "gradient_ascent", "random_noise"]

# Breakdown matrix at severe attack rate f=30%
f30_df = df[df["byzantine_rate"] == 0.3] if len(df) > 0 else pd.DataFrame()
pivot_f30 = f30_df.pivot(index="method", columns="attack", values="mean_acc") if len(f30_df) > 0 else pd.DataFrame()

formatted_f30 = pd.DataFrame(index=method_order)
for atk_id in attack_ids:
    col_vals = []
    for m in method_order:
        if atk_id in pivot_f30.columns and m in pivot_f30.index and pd.notna(pivot_f30.loc[m, atk_id]):
            col_vals.append(f"{pivot_f30.loc[m, atk_id]:.2f}")
        else:
            col_vals.append("--")
    formatted_f30[atk_id] = col_vals

print("=== Table IV: CIFAR-100 Byzantine Breakdown Matrix at f = 30% Attackers ===")
print(formatted_f30.to_markdown())

latex_code = formatted_f30.to_latex(
    caption="AAMAS 2027 Table IV: Byzantine Robustness under 4 Poisoning Strategies at $f=30\\%$ Malicious Clients (CIFAR-100, $\\alpha=0.5$).",
    label="tab:cifar100_byzantine_matrix"
)
latex_file = os.path.join(OUT_DIR, "table4_cifar100_byzantine.tex")
with open(latex_file, "w") as f:
    f.write(latex_code)
print(f"\nLaTeX code exported to: {latex_file}")


## 7. Publication Heatmaps & Robustness Degradation Curves


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 4, figsize=(22, 5), dpi=300, sharey=True)
sns.set_theme(style="whitegrid", font_scale=1.05)

attack_names = ["label_flip", "sign_flip", "gradient_ascent", "random_noise"]
attack_titles = ["Label Flipping", "Sign Flipping", "Gradient Ascent", "Gaussian Noise"]

palette = {
    "fedavg": "#7f7f7f",
    "fedprox": "#1f77b4",
    "multikrum": "#ff7f0e",
    "scaffold": "#2ca02c",
    "ditto": "#9467bd",
    "topo": "#e377c2",
    "topo_defended": "#d62728",
}

labels = {
    "fedavg": "FedAvg",
    "fedprox": "FedProx",
    "multikrum": "Multi-Krum",
    "scaffold": "SCAFFOLD",
    "ditto": "Ditto",
    "topo": "Proposed Topo (HEP)",
    "topo_defended": "Proposed Topo (Defended)",
}

for idx, (atk_id, title) in enumerate(zip(attack_names, attack_titles)):
    ax = axes[idx]
    atk_sub = df[df["attack"] == atk_id] if "attack" in df.columns else pd.DataFrame()
    
    for m_id in method_order:
        if len(atk_sub) == 0:
            continue
        m_data = atk_sub[atk_sub["method"] == m_id].sort_values("byzantine_rate")
        if len(m_data) == 0:
            continue
        
        rates = m_data["byzantine_rate"] * 100
        accs = m_data["mean_acc"]
        
        lw = 3.0 if "topo" in m_id else 1.8
        marker = "D" if m_id == "topo_defended" else "o"
        
        ax.plot(
            rates, accs,
            label=labels.get(m_id, m_id),
            color=palette.get(m_id, "#333"),
            linewidth=lw,
            marker=marker,
            markersize=6
        )
    
    ax.set_title(title, fontweight="bold", pad=10)
    ax.set_xlabel("Byzantine Attackers (%)", fontweight="bold")
    if idx == 0:
        ax.set_ylabel("Test Accuracy (%)", fontweight="bold")
    ax.set_xticks([0, 10, 20, 30, 40])
    ax.set_ylim(0, 100)

axes[0].legend(frameon=True, facecolor="white", loc="lower left")
plt.suptitle("AAMAS 2027: Byzantine Attack Degradation Matrix across Poisoning Strategies (CIFAR-100)", fontweight="bold", y=1.03)
plt.tight_layout()

fig_path = os.path.join(OUT_DIR, "figure_cifar100_byzantine_curves.png")
plt.savefig(fig_path, dpi=300)
plt.savefig(os.path.join(OUT_DIR, "figure_cifar100_byzantine_curves.pdf"))
plt.show()
plt.close()
print(f"Publication degradation curves saved to: {fig_path}")


## 8. Unified Synthesis: Merge Session 1 & Session 2 (Optional)
If you upload `results_personalization.json` from Session 1 into this session's working directory, this cell merges both sessions into a single complete report!


In [ ]:
session1_path = os.path.join(ROOT, "outputs/baselines_session1/results_personalization.json")
if not os.path.exists(session1_path):
    session1_path = "/kaggle/working/results_personalization.json"

if os.path.exists(session1_path):
    print("Found Session 1 results! Generating unified summary...")
    with open(session1_path, "r") as f:
        s1 = json.load(f)
    print(f"Session 1: {len(s1)} items | Session 2: {len(byz_data)} items")
    combined = {"personalization": s1, "byzantine": byz_data}
    combined_file = os.path.join(OUT_DIR, "unified_cifar100_baselines.json")
    with open(combined_file, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"Unified report saved to: {combined_file}")
else:
    print("ℹ️ Session 1 file not detected in workspace. Both sessions remain independently packaged.")


## 9. Export Results Package for Download


In [ ]:
import shutil

zip_name = "/kaggle/working/cifar100_baselines_session2_results" if IS_KAGGLE else "./cifar100_baselines_session2_results"
shutil.make_archive(zip_name, 'zip', OUT_DIR)
print(f"📦 Successfully packaged all Session 2 results into: {zip_name}.zip")

if IS_COLAB:
    from google.colab import files
    files.download(f"{zip_name}.zip")
